# 교과서 문제 벡터DB 재구축 (v2)

기존 파이프라인의 문제점을 수정한 버전입니다.

| | 기존 | v2 |
|---|---|---|
| 임베딩 텍스트 | `str(learning_data_info)` 통째로 (~1000자) | 문항 + 성취기준만 (~100자) |
| Bounding_Box 좌표 | 임베딩에 포함됨 | 제거 |
| 성취기준 | metadata에만 | **임베딩 텍스트에 포함** |
| 정답/선택지/해설 | 문자열 속에 파묻힘 | metadata로 즉시 접근 |
| 조회 시 파싱 | 매번 `literal_eval` 필요 | 불필요 |

**데이터 로딩부(Drive 마운트 → zip 해제 → glob → json_normalize)는 원본 노트북 그대로입니다.**

> 런타임을 **GPU**로 설정하고 시작하세요. (런타임 → 런타임 유형 변경 → T4 GPU)

## 0. 설치

In [ ]:
!pip install -q chromadb sentence-transformers pandas tqdm langchain-chroma langchain-huggingface langchain-core

## 1. Drive 마운트 & 압축 해제 *(원본 노트북과 동일)*

In [ ]:
import os, re, json, glob, zipfile
from collections import Counter

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DATA_PATH    = "/content/drive/MyDrive/Training"
EXTRACT_PATH = "/content/extracted_training"

In [ ]:
# 이미 /content/extracted_training 에 해제돼 있다면 이 셀은 건너뛰어도 됩니다.
all_files  = glob.glob(DATA_PATH + "/**/*", recursive=True)
real_files = [f for f in all_files if os.path.isfile(f)]
ZIP_FILES  = [f for f in real_files if f.endswith(".zip")]

print("zip 파일 수:", len(ZIP_FILES))
for zip_path in ZIP_FILES:
    print("압축 해제 중:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
print("압축 해제 완료")

In [ ]:
all_files  = glob.glob(EXTRACT_PATH + "/**/*", recursive=True)
real_files = [f for f in all_files if os.path.isfile(f)]
json_files = [f for f in real_files if f.endswith(".json")]

print("json 파일 수:", len(json_files))   # 원본 기준 16,248
json_files[:5]

## 2. DataFrame 생성 *(원본 노트북과 동일)*

In [ ]:
import pandas as pd
from tqdm import tqdm

dfs = []
for path in tqdm(json_files):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        temp_df = pd.json_normalize(data)
        temp_df["source_file"] = path
        dfs.append(temp_df)
    except Exception as e:
        print("로드 실패:", path, e)

df = pd.concat(dfs, ignore_index=True)
print("데이터 개수:", len(df))
df.columns.tolist()

---
## 3. 전처리 ★ 핵심 변경점 ★

원본 Cell 17의 `df["text"] = df["learning_data_info"].astype(str)` 를 대체합니다.

`learning_data_info`는 `리스트 > dict > class_info_list > dict > text_description` 3중 중첩 구조입니다.
`str()`로 통째로 문자열화하면 `Bounding_Box`, `Type_value`, 좌표 숫자가 본문의 90% 이상을 차지해
임베딩 벡터가 전부 비슷한 방향으로 쏠립니다. 여기서는 `text_description`만 골라냅니다.

In [ ]:
# ①②③④⑤ 앞에서 자르기 위한 정규식.
# (?=...) 전방탐색 = 기호를 소비하지 않고 그 "앞 위치"만 매칭 → 기호가 살아있는 채로 분리됨
CHOICE_SPLIT = re.compile(r"(?=[①②③④⑤⑥⑦⑧⑨⑩])")


def collect_texts(blocks) -> dict:
    """learning_data_info의 3중 중첩을 {class_name: [text_description, ...]}로 평탄화.
    Bounding_Box 좌표는 여기서 자연스럽게 버려진다."""
    if isinstance(blocks, str):                 # 혹시 문자열로 들어온 경우 대비
        try:
            blocks = json.loads(blocks.replace("'", '"'))
        except Exception:
            return {}
    if not isinstance(blocks, list):
        return {}

    bucket = {}
    for b in blocks:
        if not isinstance(b, dict):
            continue
        name = b.get("class_name", "")
        texts = [
            ci.get("text_description", "").strip()
            for ci in b.get("class_info_list", [])
            if isinstance(ci, dict) and ci.get("text_description", "").strip()
        ]
        if texts:
            # 같은 class_name이 여러 번 등장할 수 있으므로 덮어쓰지 말고 이어붙임
            bucket.setdefault(name, []).extend(texts)
    return bucket


def pick(bucket: dict, prefix: str) -> list:
    """'문항'으로 시작하는 모든 키를 합침 → '문항(텍스트)'+'문항(이미지)'를 한 번에.
    exact match로 하면 '문항(표)' 같은 변종이 있을 때 조용히 누락된다."""
    out = []
    for k, v in bucket.items():
        if k.startswith(prefix):
            out.extend(v)
    return out


def split_choices(text: str) -> list:
    return [p.strip() for p in CHOICE_SPLIT.split(text) if p.strip()]


def clean_standard(raw) -> str:
    """['[9수02-13] 미지수가 2개인...'] 에서 알맹이만 추출.
    2009 기준처럼 [' '] 하나만 든 빈 값은 버린다."""
    if raw is None:
        return ""
    if isinstance(raw, list):
        return " ".join(str(x).strip() for x in raw if str(x).strip())
    s = str(raw).strip()
    if s in ("[' ']", "[]", "nan", "None", ""):
        return ""
    s = re.sub(r"^\[|\]$", "", s)
    s = re.sub(r"[\"']", "", s)
    return re.sub(r"\s+", " ", s).strip()

print("전처리 함수 정의 완료")

In [ ]:
def build_record(row):
    """DataFrame 한 행 → 적재용 dict. 문항이 없으면 None(스킵)."""
    b = collect_texts(row["learning_data_info"])

    # [본문] 문항(텍스트) + 문항(이미지).
    # '문항(이미지)'의 text_description에는 수식이 LaTeX로 들어있어 이게 진짜 문제 내용임
    question = "\n".join(pick(b, "문항"))
    if not question.strip():
        return None

    # [정답] 정답(텍스트)/정답(이미지)는 같은 내용 중복 → 하나만
    answer = (pick(b, "정답") or [""])[0]

    # [선택지] 정답 1개 + 오답 덩어리를 합쳐야 전체 보기가 완성됨
    choices = split_choices(answer) + split_choices(" ".join(pick(b, "오답")))
    choices.sort(key=lambda s: s[0] if s else "")   # ①②③ 유니코드가 연속이라 정렬 가능

    explanation = "\n".join(pick(b, "해설"))

    std = (clean_standard(row.get("source_data_info.2022_achievement_standard"))
           or clean_standard(row.get("source_data_info.2015_achievement_standard")))

    # ── 임베딩 대상 텍스트 ──
    # 성취기준을 붙이는 게 핵심. "다음 연립방정식을 풀어라"라는 문항에는
    # 정작 '연립일차방정식'이라는 개념어가 없어서 개념 검색이 안 걸린다.
    doc_text = f"{question}\n\n[관련 개념] {std}" if std else question

    return {
        "text": doc_text,
        "meta": {
            # Chroma metadata는 스칼라(str/int/float/bool)만 허용 → 리스트는 JSON 문자열로
            "question":    question,
            "choices":     json.dumps(choices, ensure_ascii=False),
            "n_choices":   len(choices),
            "answer":      answer,
            "explanation": explanation,
            "standard":    std,
            "school":      str(row.get("raw_data_info.school", "")),
            "grade":       str(row.get("raw_data_info.grade", "")),
            "semester":    str(row.get("raw_data_info.semester", "")),
            "subject":     str(row.get("raw_data_info.subject", "")),
            "difficulty":  str(row.get("source_data_info.level_of_difficulty", "")),
            "ptype":       str(row.get("source_data_info.types_of_problems", "")),
            "source_file": str(row.get("source_file", "")),
        },
    }


records, skipped = [], 0
for _, row in tqdm(df.iterrows(), total=len(df)):
    r = build_record(row)
    if r:
        records.append(r)
    else:
        skipped += 1

print(f"\n파싱 완료: {len(records)}건 / 스킵 {skipped}건")

### 3-1. 품질 점검 — **적재 전에 반드시 확인**

여기서 이상이 보이면 파싱 규칙을 고친 뒤 적재하세요. 한번 넣고 나면 다시 빼기 번거롭습니다.

- **임베딩 텍스트 평균 길이**: 기존 1000자+ → 100자 내외로 줄어야 정상
- **선택지 개수 분포**: 객관식인데 5가 아닌 게 많으면 `split_choices` 수정 필요
- **성취기준 결측**: 너무 많으면 개념 검색 효과가 떨어짐

In [ ]:
lens = [len(r["text"]) for r in records]
print(f"임베딩 텍스트 평균 길이 : {sum(lens)/len(lens):.0f}자  (min {min(lens)} / max {max(lens)})")
print(f"기존 str() 방식 평균    : {df['learning_data_info'].astype(str).str.len().mean():.0f}자")
print()
print("문제 유형        :", Counter(r["meta"]["ptype"] for r in records).most_common())
print("선택지 개수 분포 :", Counter(r["meta"]["n_choices"] for r in records).most_common())
print("성취기준 결측    :", sum(1 for r in records if not r["meta"]["standard"]), "건")
print("학교급           :", Counter(r["meta"]["school"] for r in records).most_common())

In [ ]:
# 객관식인데 선택지가 5개가 아닌 케이스 눈으로 확인
bad = [r for r in records if r["meta"]["ptype"] == "객관식" and r["meta"]["n_choices"] != 5]
print(f"객관식 중 선택지 5개 아닌 것: {len(bad)}건\n")

for r in bad[:3]:
    print("문항 :", r["meta"]["question"][:80])
    print("정답 :", r["meta"]["answer"])
    print("보기 :", json.loads(r["meta"]["choices"]))
    print("-" * 60)

In [ ]:
# 정상 샘플 확인
r = records[0]
print("[임베딩되는 텍스트]")
print(r["text"])
print("\n[metadata]")
for k, v in r["meta"].items():
    print(f"  {k:12s}: {str(v)[:70]}")

---
## 4. 임베딩 & 적재

원본과 동일한 모델(`jhgan/ko-sroberta-multitask`)을 사용합니다.

> **주의**: 적재할 때와 검색할 때의 임베딩 모델이 다르면 벡터 공간이 달라져
> 검색 결과가 무작위에 가까워집니다. 서비스 코드에서도 반드시 같은 모델을 쓰세요.

In [ ]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb, torch

MODEL_NAME  = "jhgan/ko-sroberta-multitask"
NEW_DB_PATH = "/content/drive/MyDrive/chroma_problem_db_v2"   # 기존 DB와 분리 (비교용)
COLLECTION  = "problem_collection"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64},
)

In [ ]:
client = chromadb.PersistentClient(path=NEW_DB_PATH)

# 재실행 시 중복 적재를 막기 위해 기존 컬렉션 삭제
try:
    client.delete_collection(COLLECTION)
    print("기존 컬렉션 삭제됨")
except Exception:
    print("기존 컬렉션 없음 (신규 생성)")

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION,
    embedding_function=embeddings,
)

In [ ]:
BATCH = 200

docs = [Document(page_content=r["text"], metadata=r["meta"]) for r in records]
ids  = [f"problem_{i}" for i in range(len(docs))]

for i in tqdm(range(0, len(docs), BATCH)):
    vectorstore.add_documents(docs[i:i+BATCH], ids=ids[i:i+BATCH])

print("\n저장 개수:", vectorstore._collection.count())
print("저장 위치:", NEW_DB_PATH)

---
## 5. 검증

재구축 효과를 눈으로 확인합니다.
개념어로 검색했을 때 **해당 단원 문제만** 나와야 정상입니다.

In [ ]:
def show(query, k=3, **flt):
    key_map = {"grade":"grade", "difficulty":"difficulty", "ptype":"ptype", "school":"school"}
    conds = [{key_map[k_]: v} for k_, v in flt.items() if v and k_ in key_map]
    where = conds[0] if len(conds) == 1 else ({"$and": conds} if conds else None)

    print(f"\n{'='*65}\n[query] {query}   {flt if flt else ''}")
    for d in vectorstore.similarity_search(query, k=k, filter=where):
        m = d.metadata
        print(f"\n  {m['school']} {m['grade']} {m['semester']} / 난이도 {m['difficulty']} / {m['ptype']}")
        print(f"  기준: {m['standard'][:55]}")
        print(f"  문항: {m['question'][:70]}")
        print(f"  보기: {json.loads(m['choices'])}")


show("연립일차방정식 대입법")
show("비례식의 성질")
show("원의 넓이 구하기")

In [ ]:
# 메타데이터 필터 동작 확인
show("일차방정식", k=3, school="중학교", difficulty="하")